In [1]:
!pip install lightgbm -q

from google.colab import files
uploaded = files.upload()

from google.cloud import bigquery
client = bigquery.Client.from_service_account_json(list(uploaded.keys())[0])
print("Connected to project:", client.project)

import numpy as np
import pandas as pd
import lightgbm as lgb

# ---- 1. Load weekly mart + calendar from BigQuery ----
mart = client.query("""
SELECT store_id, cat_id, dept_id, wm_yr_wk, total_units_sold AS y
FROM `phonic-chemist-509603-v9.m5_analytics.mart_weekly_sales`
""").to_dataframe()

cal = client.query("""
SELECT wm_yr_wk, MIN(date) AS week_start, COUNT(*) AS days,
       COUNTIF(event_name_1 IS NOT NULL) AS n_events,
       SUM(snap_CA) AS snap_CA, SUM(snap_TX) AS snap_TX, SUM(snap_WI) AS snap_WI
FROM `phonic-chemist-509603-v9.m5_raw.calendar`
GROUP BY wm_yr_wk
""").to_dataframe()
cal["week_start"] = pd.to_datetime(cal["week_start"])
mart["wm_yr_wk"] = mart["wm_yr_wk"].astype(int)
cal["wm_yr_wk"] = cal["wm_yr_wk"].astype(int)

# ---- 2. History (drop last partial sales week) + 4 future weeks ----
hist = mart.merge(cal[["wm_yr_wk", "week_start"]], on="wm_yr_wk", how="left")
hist = hist[hist["week_start"] < hist["week_start"].max()].copy()
last_week = hist["week_start"].max()

keys = hist[["store_id", "cat_id", "dept_id"]].drop_duplicates()
fut_weeks = cal[(cal["week_start"] > last_week) & (cal["days"] == 7)].sort_values("week_start").head(4)
future = keys.merge(fut_weeks[["wm_yr_wk", "week_start"]], how="cross")
future["y"] = np.nan

full = pd.concat([hist[["store_id", "cat_id", "dept_id", "wm_yr_wk", "week_start", "y"]], future], ignore_index=True)
full = full.merge(cal[["wm_yr_wk", "n_events", "snap_CA", "snap_TX", "snap_WI"]], on="wm_yr_wk", how="left")
state = full["store_id"].str[:2]
full["snap_days"] = np.select([state == "CA", state == "TX"], [full["snap_CA"], full["snap_TX"]], default=full["snap_WI"])
full = full.sort_values(["store_id", "dept_id", "week_start"]).reset_index(drop=True)

# ---- 3. Features (all lags >= 12 weeks, so 4 future weeks need no recursion) ----
g = full.groupby(["store_id", "dept_id"])["y"]
for lag in [12, 13, 26, 52]:
    full[f"lag_{lag}"] = g.shift(lag)
full["roll_mean_4_lag12"] = g.transform(lambda s: s.shift(12).rolling(4).mean())
full["roll_mean_13_lag12"] = g.transform(lambda s: s.shift(12).rolling(13).mean())
full["roll_std_8_lag12"] = g.transform(lambda s: s.shift(12).rolling(8).std())
full["weekofyear"] = full["week_start"].dt.isocalendar().week.astype(int)
full["month"] = full["week_start"].dt.month
for c in ["store_id", "cat_id", "dept_id"]:
    full[c] = full[c].astype("category")

features = ["store_id", "cat_id", "dept_id", "weekofyear", "month", "lag_12", "lag_13", "lag_26", "lag_52",
            "roll_mean_4_lag12", "roll_mean_13_lag12", "roll_std_8_lag12", "n_events", "snap_days"]

# If you have lightgbm_best_params.json from Day 14, paste its values here
params = dict(num_leaves=31, learning_rate=0.05, min_child_samples=20)

# ---- 4. Holdout check (last 12 weeks) ----
known = full[full["y"].notna()].dropna(subset=features)
test_start = sorted(hist["week_start"].unique())[-12]
tr, te = known[known["week_start"] < test_start], known[known["week_start"] >= test_start].copy()
m1 = lgb.LGBMRegressor(n_estimators=400, random_state=42, verbose=-1, **params).fit(tr[features], tr["y"])
te["forecast"] = m1.predict(te[features])
print("Holdout WMAPE:", round(np.abs(te["y"] - te["forecast"]).sum() / te["y"].sum(), 4))

# ---- 5. Final model on all history -> next 4 weeks ----
m2 = lgb.LGBMRegressor(n_estimators=400, random_state=42, verbose=-1, **params).fit(known[features], known["y"])
fut = full[full["y"].isna()].copy()
fut["forecast"] = m2.predict(fut[features])
print("Future weeks:", sorted(fut["week_start"].dt.date.unique()))

# ---- 6. Combine + save ----
te["type"], te["actual"] = "holdout", te["y"]
fut["type"], fut["actual"] = "future", np.nan
out = pd.concat([te, fut])[["type", "week_start", "store_id", "cat_id", "dept_id", "actual", "forecast"]].copy()
for c in ["store_id", "cat_id", "dept_id"]:
    out[c] = out[c].astype(str)
out["forecast"] = out["forecast"].clip(lower=0).round(1)
out["week_start"] = out["week_start"].dt.date

history = hist[["store_id", "cat_id", "dept_id", "week_start", "y"]].rename(columns={"y": "units"}).copy()
history["week_start"] = history["week_start"].dt.date

out.to_csv("forecasts.csv", index=False)
history.to_csv("weekly_history.csv", index=False)
print("Saved forecasts.csv", out.shape, "| weekly_history.csv", history.shape)

# ---- 7. Write back to BigQuery (small tables, fits the free quota) ----
try:
    client.load_table_from_dataframe(out, "phonic-chemist-509603-v9.m5_analytics.demand_forecasts").result()
    client.load_table_from_dataframe(history, "phonic-chemist-509603-v9.m5_analytics.weekly_history").result()
    print("Loaded demand_forecasts and weekly_history to BigQuery")
except Exception as e:
    print("BigQuery load failed (CSVs are still saved):", e)

Saving phonic-chemist-509603-v9-6d0523a7b77d.json to phonic-chemist-509603-v9-6d0523a7b77d.json
Connected to project: phonic-chemist-509603-v9


/tmp/ipykernel_1240/2480314271.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full = pd.concat([hist[["store_id", "cat_id", "dept_id", "wm_yr_wk", "week_start", "y"]], future], ignore_index=True)


Holdout WMAPE: 0.1029
Future weeks: [datetime.date(2016, 4, 23), datetime.date(2016, 4, 30), datetime.date(2016, 5, 7), datetime.date(2016, 5, 14)]
Saved forecasts.csv (1120, 7) | weekly_history.csv (19110, 5)


/tmp/ipykernel_1240/2480314271.py:82: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat([te, fut])[["type", "week_start", "store_id", "cat_id", "dept_id", "actual", "forecast"]].copy()


Loaded demand_forecasts and weekly_history to BigQuery
